# Notebook 5 — Gemma via Ollama vs Gemini Flash

**What this tests:**
- Run the same 8 agent tasks through both models
- Score: response quality, instruction following, JSON validity, Spanish accuracy, latency, token cost
- Specific test: does Gemma follow the context.md structure rules as reliably as Gemini?
- Output: comparison table Harshal can show Durgesh for model decision

**Setup:**
1. Install Ollama on your local machine: `curl -fsSL https://ollama.ai/install.sh | sh`
2. Pull Gemma: `ollama pull gemma3:4b` (4B param version, runs on most laptops)
3. Start Ollama: `ollama serve`
4. Add `GEMINI_API_KEY` to Colab Secrets
5. Run this notebook — it calls Ollama at localhost:11434 and Gemini API

**Note:** Ollama must be running locally. This notebook uses ngrok or localtunnel
to expose the local Ollama endpoint to Colab if needed.

In [ ]:
!pip install -q google-generativeai requests
import google.generativeai as genai
import requests, json, time
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# ── Ollama config ─────────────────────────────────────
# If running Colab with local Ollama, use ngrok:
#   ngrok http 11434
# Then paste the ngrok URL below.
# If running this notebook LOCALLY (not Colab), use localhost.

OLLAMA_URL = 'http://localhost:11434'  # change to ngrok URL if using Colab
GEMMA_MODEL = 'gemma3:4b'             # or gemma3:12b if you have more RAM

def check_ollama():
    try:
        r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
        models = [m['name'] for m in r.json().get('models', [])]
        if GEMMA_MODEL in models:
            print(f'✅ Ollama running. {GEMMA_MODEL} is available.')
            return True
        else:
            print(f'⚠️  Ollama running but {GEMMA_MODEL} not found.')
            print(f'   Available: {models}')
            print(f'   Run: ollama pull {GEMMA_MODEL}')
            return False
    except Exception as e:
        print(f'❌ Ollama not reachable at {OLLAMA_URL}')
        print(f'   Start with: ollama serve')
        print(f'   If using Colab, set OLLAMA_URL to your ngrok URL')
        return False

ollama_ok = check_ollama()

In [ ]:
# ── Model wrappers ────────────────────────────────────

gemini_model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    generation_config={'temperature': 0.1, 'max_output_tokens': 800}
)

def call_gemini(prompt):
    start = time.time()
    response = gemini_model.generate_content(prompt)
    latency = time.time() - start
    text = response.text.strip()
    # Gemini returns usage_metadata
    usage = response.usage_metadata
    return {
        'text': text,
        'latency_s': round(latency, 2),
        'input_tokens': getattr(usage, 'prompt_token_count', 0),
        'output_tokens': getattr(usage, 'candidates_token_count', 0),
    }

def call_gemma(prompt):
    if not ollama_ok:
        return {'text': '[Ollama not available]', 'latency_s': 0,
                'input_tokens': 0, 'output_tokens': 0}
    start = time.time()
    r = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json={'model': GEMMA_MODEL, 'prompt': prompt, 'stream': False},
        timeout=120
    )
    latency = time.time() - start
    data = r.json()
    return {
        'text': data.get('response', '').strip(),
        'latency_s': round(latency, 2),
        'input_tokens': data.get('prompt_eval_count', 0),
        'output_tokens': data.get('eval_count', 0),
    }

print('✅ Model wrappers ready')

In [ ]:
# ── Test suite — 8 tasks covering all query types ─────

CONTEXT_MD = """
# Citizen Context — María
## Profile [Layer 1]
Life event: new-baby | Employment: employed (formal) | Location: Soyapango
## Situation [Layer 2]
Urgency: HIGH — RNPN deadline in 9 days
Blockers: birth certificate not yet collected
Prior attempts: Tried Simple SV, got confused by login
## Sentiment [Layer 3]
Frustration: medium | Trust: building
""".strip()

SYSTEM = f"""
You are a government services assistant for El Salvador citizens.
Always respond in the same language the citizen uses.
Never invent government services. Be concise and actionable.
Citizen context:
{CONTEXT_MD}
"""

TEST_CASES = [
    {
        "id": "T1_service_lookup",
        "type": "service-lookup",
        "prompt": SYSTEM + "\n\nCitizen: What do I need to do to register my baby?",
        "checks": ["RNPN", "deadline", "birth certificate"],
        "must_not_contain": ["CONAMYPE", "INSAFORP"],
        "language": "english"
    },
    {
        "id": "T2_spanish",
        "type": "spanish-response",
        "prompt": SYSTEM + "\n\nCiudadana: ¿Qué documentos necesito para el RNPN?",
        "checks": ["DUI", "nacimiento"],
        "must_not_contain": [],
        "language": "spanish"
    },
    {
        "id": "T3_urgency_aware",
        "type": "urgency-acknowledgement",
        "prompt": SYSTEM + "\n\nCitizen: What should I do first?",
        "checks": ["RNPN", "9"],  # should mention 9 days
        "must_not_contain": [],
        "language": "english"
    },
    {
        "id": "T4_blocker_aware",
        "type": "blocker-acknowledgement",
        "prompt": SYSTEM + "\n\nCitizen: I'm ready to start the RNPN registration now.",
        "checks": ["birth certificate"],  # should flag missing document
        "must_not_contain": [],
        "language": "english"
    },
    {
        "id": "T5_context_md_update",
        "type": "context-update",
        "prompt": f"""
Update this context.md based on the new turn.
User said: "I just collected the birth certificate!"
Agent said: "That's the main blocker cleared. You're ready for RNPN registration."

Current context.md:
{CONTEXT_MD}

Return ONLY the updated context.md. Keep all sections. Remove resolved blockers.
""",
        "checks": ["Profile", "Situation", "Sentiment"],
        "must_not_contain": ["birth certificate not yet collected"],
        "language": "structure"
    },
    {
        "id": "T6_no_hallucination",
        "type": "hallucination-guard",
        "prompt": SYSTEM + "\n\nCitizen: Is there a $500 government cash grant for new mothers?",
        "checks": ["not"],  # should say it doesn't know or can't confirm
        "must_not_contain": ["$500", "grant"],
        "language": "english"
    },
    {
        "id": "T7_out_of_scope",
        "type": "out-of-scope",
        "prompt": SYSTEM + "\n\nCitizen: Can you help me book a flight to Miami?",
        "checks": ["government", "service"],  # should redirect
        "must_not_contain": ["flight", "ticket", "airline"],
        "language": "english"
    },
    {
        "id": "T8_one_question_rule",
        "type": "one-question-rule",
        "prompt": SYSTEM + "\n\nCitizen: I need help.",
        "checks": ["?"],  # should ask exactly one question
        "must_not_contain": [],
        "language": "english",
        "max_questions": 1
    },
]

print(f'✅ {len(TEST_CASES)} test cases loaded')
for tc in TEST_CASES:
    print(f'  [{tc["id"]}] — {tc["type"]}')

In [ ]:
# ── Run all tests through both models ─────────────────

def score_response(response_text, checks, must_not_contain, max_questions=None):
    score = 0
    total = len(checks) + len(must_not_contain)
    details = []

    for check in checks:
        found = check.lower() in response_text.lower()
        if found: score += 1
        details.append(('✅' if found else '❌', f'contains "{check}"'))

    for term in must_not_contain:
        absent = term.lower() not in response_text.lower()
        if absent: score += 1
        details.append(('✅' if absent else '❌', f'does NOT contain "{term}"'))

    if max_questions is not None:
        q_count = response_text.count('?')
        ok = q_count <= max_questions
        if ok: score += 1
        total += 1
        details.append(('✅' if ok else '❌', f'question count {q_count} <= {max_questions}'))

    return score, total, details

results = []

for tc in TEST_CASES:
    print(f'\nRunning [{tc["id"]}]...')

    g_result = call_gemini(tc['prompt'])
    m_result = call_gemma(tc['prompt'])

    g_score, total, g_details = score_response(
        g_result['text'], tc['checks'], tc['must_not_contain'],
        tc.get('max_questions')
    )
    m_score, _, m_details = score_response(
        m_result['text'], tc['checks'], tc['must_not_contain'],
        tc.get('max_questions')
    )

    results.append({
        'id': tc['id'],
        'type': tc['type'],
        'gemini': {**g_result, 'score': g_score, 'total': total, 'details': g_details},
        'gemma':  {**m_result, 'score': m_score, 'total': total, 'details': m_details},
    })

    print(f'  Gemini: {g_score}/{total} | {g_result["latency_s"]}s | {g_result["input_tokens"]+g_result["output_tokens"]} tokens')
    print(f'  Gemma:  {m_score}/{total} | {m_result["latency_s"]}s | {m_result["input_tokens"]+m_result["output_tokens"]} tokens')

print('\n✅ All tests complete')

In [ ]:
# ── Comparison table ──────────────────────────────────

print(f'\n{"Test":<30} {"Gemini Score":>12} {"Gemma Score":>12} {"Gemini ms":>10} {"Gemma ms":>10} {"Winner":>8}')
print('─' * 90)

gemini_wins = gemma_wins = ties = 0
gemini_total_tokens = gemma_total_tokens = 0
gemini_total_latency = gemma_total_latency = 0

for r in results:
    gs = r['gemini']['score']; gt = r['gemini']['total']
    ms = r['gemma']['score'];  mt = r['gemma']['total']
    gl = r['gemini']['latency_s']; ml = r['gemma']['latency_s']

    if gs > ms:   winner = 'Gemini'; gemini_wins += 1
    elif ms > gs: winner = 'Gemma';  gemma_wins += 1
    else:         winner = 'Tie';    ties += 1

    gemini_total_tokens += r['gemini']['input_tokens'] + r['gemini']['output_tokens']
    gemma_total_tokens  += r['gemma']['input_tokens']  + r['gemma']['output_tokens']
    gemini_total_latency += gl; gemma_total_latency += ml

    print(f"{r['id']:<30} {f'{gs}/{gt}':>12} {f'{ms}/{mt}':>12} {f'{gl}s':>10} {f'{ml}s':>10} {winner:>8}")

print('─' * 90)
print(f'TOTALS: Gemini wins={gemini_wins} | Gemma wins={gemma_wins} | Ties={ties}')
print(f'Avg latency: Gemini={gemini_total_latency/len(results):.2f}s | Gemma={gemma_total_latency/len(results):.2f}s')
print(f'Total tokens: Gemini={gemini_total_tokens} | Gemma={gemma_total_tokens}')
print(f'\nNote: Gemma token cost = $0 (local). Gemini cost = ~${gemini_total_tokens * 0.000001:.4f} for this test run.')

In [ ]:
# ── Deep dive: context.md update test (T5) ────────────
# Most important test — does Gemma follow structural rules?

t5_gemini = next(r for r in results if r['id'] == 'T5_context_md_update')

print('T5 — context.md update')
print('='*60)
print('GEMINI output:')
print(t5_gemini['gemini']['text'])
print('\nGEMMA output:')
print(t5_gemini['gemma']['text'])
print('\nChecks:')
for icon, detail in t5_gemini['gemini']['details']:
    print(f'  Gemini {icon} {detail}')
for icon, detail in t5_gemini['gemma']['details']:
    print(f'  Gemma  {icon} {detail}')

In [ ]:
# ── Recommendation ────────────────────────────────────
g_total = sum(r['gemini']['score'] for r in results)
m_total = sum(r['gemma']['score'] for r in results)
g_max   = sum(r['gemini']['total'] for r in results)

g_pct = g_total / g_max * 100
m_pct = m_total / g_max * 100
diff  = g_pct - m_pct

print(f"""
OVERALL SCORES
──────────────────────────────────────────
Gemini 2.0 Flash:  {g_total}/{g_max} ({g_pct:.0f}%)
Gemma 3 4B local:  {m_total}/{g_max} ({m_pct:.0f}%)
Quality gap:       {diff:+.0f} percentage points

RECOMMENDATION
──────────────────────────────────────────
Use case: Turn processor (context.md updates)
  → Use whichever scores higher on T5
  → This is a structured task — Gemma may be competitive

Use case: Main chat response (streaming)
  → Needs streaming support + Spanish quality
  → Check T2 (Spanish) score carefully

Use case: Query classifier
  → Small, fast, deterministic — Gemma 4B is ideal
  → If Gemma T1/T6/T7 scores are high, use it here

Cost implication:
  → Gemma local = $0/token, ~{gemma_total_latency/len(results):.1f}s avg latency
  → Gemini = cost per token, ~{gemini_total_latency/len(results):.1f}s avg latency
  → Hybrid: Gemma for classifier + turn processor, Gemini for main chat
""")